# 🗄️ Vector Databases - Scaling RAG to Production

## What are Vector Databases?

**Vector databases are specialized databases optimized for storing and querying high-dimensional vectors (embeddings) at scale.**

### The Problem They Solve:

```python
# Naive approach (doesn't scale)
for doc in 1_000_000_documents:  # Too slow!
    similarity = cosine_similarity(query, doc)

# Vector DB approach (fast!)
results = vector_db.search(query, top_k=10)  # Milliseconds!
```

### The Magic:
- **ANN (Approximate Nearest Neighbors)**: Find similar vectors FAST
- **Indexing**: Organize vectors for quick retrieval
- **Scalability**: Handle millions or billions of vectors

![Vector Database Architecture](./images/vector_db_architecture.png)
*Figure 1: How vector databases index and query embeddings*

## Popular Vector Databases

### Cloud/Managed:
1. **Pinecone** ⭐ - Easiest to use, fully managed
2. **Weaviate** - Open-source, feature-rich
3. **Qdrant** - Rust-based, high performance
4. **Milvus** - Scalable, open-source

### Local/Embedded:
1. **ChromaDB** ⭐ - Simple, embedded, great for prototyping
2. **FAISS** - Facebook AI, lightning fast
3. **Annoy** - Spotify, memory efficient
4. **LanceDB** - Serverless, modern

### Hybrid (Vector + Traditional):
1. **Elasticsearch** - Vector + text search
2. **PostgreSQL (pgvector)** - SQL + vectors
3. **Redis** - In-memory + vectors

## Key Concepts

### 1. Indexing Methods
- **HNSW** (Hierarchical Navigable Small World): Best quality
- **IVF** (Inverted File): Fast, good for large datasets
- **LSH** (Locality Sensitive Hashing): Memory efficient
- **PQ** (Product Quantization): Compression

### 2. Similarity Metrics
- **Cosine**: Most common
- **Euclidean**: L2 distance
- **Dot Product**: Fast if normalized

### 3. Trade-offs
```
Speed ←→ Accuracy
Memory ←→ Quality
Simple ←→ Scalable
```

## 1. ChromaDB - Simple & Powerful

In [1]:
# Install ChromaDB
# !pip install chromadb sentence-transformers

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB (in-memory)
client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# Create collection
collection = client.create_collection(
    name="rag_documents",
    metadata={"description": "RAG knowledge base"}
)

print("✅ ChromaDB initialized!")
print(f"Collection: {collection.name}")

✅ ChromaDB initialized!
Collection: rag_documents


In [2]:
# Sample documents
documents = [
    "Python is a versatile programming language used in data science.",
    "Machine learning models learn patterns from data.",
    "Natural language processing enables AI to understand text.",
    "Deep learning uses neural networks with multiple layers.",
    "RAG systems combine retrieval with language generation.",
    "Vector databases store embeddings for fast similarity search.",
    "Transformers revolutionized natural language understanding.",
    "Embeddings convert text into numerical representations.",
]

# Create IDs and metadata
ids = [f"doc_{i}" for i in range(len(documents))]
metadata = [{"source": "knowledge_base", "index": i} for i in range(len(documents))]

# Add to ChromaDB (it auto-generates embeddings!)
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadata
)

print(f"✅ Added {len(documents)} documents to ChromaDB")
print(f"Collection count: {collection.count()}")

✅ Added 8 documents to ChromaDB
Collection count: 8


In [ ]:
# Query ChromaDB
query = "How does machine learning work?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print(f"Query: '{query}'\n")
print("Top Results from ChromaDB:")
print("="*80)

for i, (doc, distance, metadata) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\n{i+1}. [Distance: {distance:.4f}]")
    print(f"   {doc}")
    print(f"   Metadata: {metadata}")

print("\n💡 ChromaDB auto-generated embeddings and performed similarity search!")

Query: 'How does machine learning work?'

Top Results from ChromaDB:

1. [Distance: 0.6550]
   Machine learning models learn patterns from data.
   Metadata: {'index': 1, 'source': 'knowledge_base'}

2. [Distance: 1.0840]
   Deep learning uses neural networks with multiple layers.
   Metadata: {'source': 'knowledge_base', 'index': 3}

3. [Distance: 1.2660]
   Natural language processing enables AI to understand text.
   Metadata: {'source': 'knowledge_base', 'index': 2}

💡 ChromaDB auto-generated embeddings and performed similarity search!


## 2. FAISS - Facebook's Lightning-Fast Library

In [4]:
# Install FAISS
# !pip install faiss-cpu  # or faiss-gpu for GPU

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings
embeddings = model.encode(documents)
dimension = embeddings.shape[1]

print(f"Embeddings shape: {embeddings.shape}")
print(f"Dimension: {dimension}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Embeddings shape: (8, 384)
Dimension: 384


In [5]:
# Create FAISS index (Flat - exact search)
index_flat = faiss.IndexFlatL2(dimension)
index_flat.add(embeddings.astype('float32'))

print(f"✅ FAISS Flat Index created")
print(f"   Vectors in index: {index_flat.ntotal}")
print(f"   Index type: Exact search (L2 distance)")

✅ FAISS Flat Index created
   Vectors in index: 8
   Index type: Exact search (L2 distance)


In [6]:
# Search with FAISS
query = "machine learning and AI"
query_embedding = model.encode([query]).astype('float32')

# Search for top-3
distances, indices = index_flat.search(query_embedding, k=3)

print(f"Query: '{query}'\n")
print("FAISS Results:")
print("="*80)

for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    print(f"\n{i+1}. [L2 Distance: {dist:.4f}]")
    print(f"   {documents[idx]}")

print("\n💡 FAISS provides exact nearest neighbor search!")

Query: 'machine learning and AI'

FAISS Results:

1. [L2 Distance: 0.8705]
   Machine learning models learn patterns from data.

2. [L2 Distance: 1.0715]
   Natural language processing enables AI to understand text.

3. [L2 Distance: 1.1449]
   Deep learning uses neural networks with multiple layers.

💡 FAISS provides exact nearest neighbor search!


In [7]:
# Advanced: FAISS with HNSW (faster for large datasets)
M = 32  # Number of connections
efConstruction = 40  # Construction time/quality trade-off

index_hnsw = faiss.IndexHNSWFlat(dimension, M)
index_hnsw.hnsw.efConstruction = efConstruction
index_hnsw.add(embeddings.astype('float32'))

# Search
distances_hnsw, indices_hnsw = index_hnsw.search(query_embedding, k=3)

print("FAISS HNSW Index:")
print(f"  M (connections): {M}")
print(f"  efConstruction: {efConstruction}")
print(f"  Vectors indexed: {index_hnsw.ntotal}")

print("\nHNSW Results:")
for i, (idx, dist) in enumerate(zip(indices_hnsw[0], distances_hnsw[0])):
    print(f"{i+1}. [Distance: {dist:.4f}] {documents[idx][:60]}...")

print("\n💡 HNSW is approximate but much faster for millions of vectors!")

FAISS HNSW Index:
  M (connections): 32
  efConstruction: 40
  Vectors indexed: 8

HNSW Results:
1. [Distance: 0.8705] Machine learning models learn patterns from data....
2. [Distance: 1.0715] Natural language processing enables AI to understand text....
3. [Distance: 1.1449] Deep learning uses neural networks with multiple layers....

💡 HNSW is approximate but much faster for millions of vectors!


## 3. Complete Vector DB Wrapper

In [8]:
from typing import List, Dict, Optional
import pickle

class VectorDatabase:
    def __init__(self, 
                 embedding_model='all-MiniLM-L6-v2',
                 index_type='hnsw',
                 metric='cosine'):
        """
        Vector Database wrapper using FAISS
        
        Parameters:
        - index_type: 'flat' (exact) or 'hnsw' (approximate)
        - metric: 'cosine', 'l2', or 'dot'
        """
        self.model = SentenceTransformer(embedding_model)
        self.index_type = index_type
        self.metric = metric
        
        self.documents = []
        self.metadata = []
        self.index = None
        self.dimension = self.model.get_sentence_embedding_dimension()
        
    def _create_index(self, embeddings):
        """Create FAISS index"""
        if self.metric == 'cosine':
            # Normalize for cosine similarity
            faiss.normalize_L2(embeddings)
            
        if self.index_type == 'flat':
            if self.metric == 'cosine' or self.metric == 'dot':
                self.index = faiss.IndexFlatIP(self.dimension)  # Inner product
            else:
                self.index = faiss.IndexFlatL2(self.dimension)  # L2 distance
        
        elif self.index_type == 'hnsw':
            M = 32
            if self.metric == 'cosine' or self.metric == 'dot':
                self.index = faiss.IndexHNSWFlat(self.dimension, M, faiss.METRIC_INNER_PRODUCT)
            else:
                self.index = faiss.IndexHNSWFlat(self.dimension, M)
            self.index.hnsw.efConstruction = 40
        
        self.index.add(embeddings)
    
    def add(self, documents: List[str], metadata: Optional[List[Dict]] = None):
        """Add documents to index"""
        self.documents.extend(documents)
        self.metadata.extend(metadata if metadata else [{} for _ in documents])
        
        # Encode
        embeddings = self.model.encode(
            documents,
            batch_size=32,
            show_progress_bar=True
        ).astype('float32')
        
        # Create or update index
        if self.index is None:
            self._create_index(embeddings)
        else:
            if self.metric == 'cosine':
                faiss.normalize_L2(embeddings)
            self.index.add(embeddings)
        
        print(f"✅ Added {len(documents)} documents (total: {self.index.ntotal})")
    
    def search(self, query: str, top_k: int = 5, score_threshold: float = None):
        """Search for similar documents"""
        # Encode query
        query_embedding = self.model.encode([query]).astype('float32')
        
        if self.metric == 'cosine':
            faiss.normalize_L2(query_embedding)
        
        # Search
        distances, indices = self.index.search(query_embedding, top_k)
        
        # Build results
        results = []
        for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
            # Convert distance to similarity score for cosine
            if self.metric == 'cosine' or self.metric == 'dot':
                score = float(dist)  # Inner product (higher is better)
            else:
                score = 1 / (1 + float(dist))  # Convert L2 to similarity
            
            # Apply threshold
            if score_threshold and score < score_threshold:
                continue
            
            results.append({
                'rank': rank + 1,
                'score': score,
                'distance': float(dist),
                'document': self.documents[idx],
                'metadata': self.metadata[idx],
                'index': int(idx)
            })
        
        return results
    
    def save(self, path: str):
        """Save index and data"""
        # Save FAISS index
        faiss.write_index(self.index, f"{path}.index")
        
        # Save metadata
        with open(f"{path}.pkl", 'wb') as f:
            pickle.dump({
                'documents': self.documents,
                'metadata': self.metadata,
                'config': {
                    'index_type': self.index_type,
                    'metric': self.metric,
                    'dimension': self.dimension
                }
            }, f)
        
        print(f"✅ Saved to {path}.index and {path}.pkl")
    
    def load(self, path: str):
        """Load index and data"""
        # Load FAISS index
        self.index = faiss.read_index(f"{path}.index")
        
        # Load metadata
        with open(f"{path}.pkl", 'rb') as f:
            data = pickle.load(f)
        
        self.documents = data['documents']
        self.metadata = data['metadata']
        config = data['config']
        self.index_type = config['index_type']
        self.metric = config['metric']
        self.dimension = config['dimension']
        
        print(f"✅ Loaded {len(self.documents)} documents from {path}")

# Test the vector database
vdb = VectorDatabase(index_type='hnsw', metric='cosine')

# Add documents
metadata = [{"source": "kb", "index": i} for i in range(len(documents))]
vdb.add(documents, metadata=metadata)

# Search
results = vdb.search("machine learning and AI", top_k=3)

print("\nVector Database Search:")
for r in results:
    print(f"\n{r['rank']}. [Score: {r['score']:.4f}]")
    print(f"   {r['document']}")
    print(f"   {r['metadata']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Added 8 documents (total: 8)

Vector Database Search:

1. [Score: 0.5647]
   Machine learning models learn patterns from data.
   {'source': 'kb', 'index': 1}

2. [Score: 0.4642]
   Natural language processing enables AI to understand text.
   {'source': 'kb', 'index': 2}

3. [Score: 0.4275]
   Deep learning uses neural networks with multiple layers.
   {'source': 'kb', 'index': 3}


## 4. Comparing Vector Databases

In [9]:
import time

# Create larger dataset
large_docs = documents * 100  # 800 documents
query = "machine learning and artificial intelligence"

print(f"Dataset size: {len(large_docs)} documents")
print(f"Query: '{query}'\n")
print("="*80)

# Test FAISS Flat (exact)
vdb_flat = VectorDatabase(index_type='flat', metric='cosine')
start = time.time()
vdb_flat.add(large_docs)
index_time_flat = time.time() - start

start = time.time()
results_flat = vdb_flat.search(query, top_k=5)
search_time_flat = time.time() - start

# Test FAISS HNSW (approximate)
vdb_hnsw = VectorDatabase(index_type='hnsw', metric='cosine')
start = time.time()
vdb_hnsw.add(large_docs)
index_time_hnsw = time.time() - start

start = time.time()
results_hnsw = vdb_hnsw.search(query, top_k=5)
search_time_hnsw = time.time() - start

# Test ChromaDB
client_test = chromadb.Client(Settings(anonymized_telemetry=False))
collection_test = client_test.create_collection(name="test")

start = time.time()
collection_test.add(
    documents=large_docs,
    ids=[f"doc_{i}" for i in range(len(large_docs))]
)
index_time_chroma = time.time() - start

start = time.time()
results_chroma = collection_test.query(query_texts=[query], n_results=5)
search_time_chroma = time.time() - start

# Print comparison
print("\nPerformance Comparison:")
print(f"{'Method':<20} {'Index Time':<15} {'Search Time':<15} {'Quality'}")
print("="*70)
print(f"{'FAISS Flat':<20} {index_time_flat:<15.3f}s {search_time_flat*1000:<15.1f}ms Exact")
print(f"{'FAISS HNSW':<20} {index_time_hnsw:<15.3f}s {search_time_hnsw*1000:<15.1f}ms ~99%")
print(f"{'ChromaDB':<20} {index_time_chroma:<15.3f}s {search_time_chroma*1000:<15.1f}ms ~99%")

print("\n💡 HNSW is much faster for search with minimal quality loss!")

Dataset size: 800 documents
Query: 'machine learning and artificial intelligence'



Batches:   0%|          | 0/25 [00:00<?, ?it/s]

✅ Added 800 documents (total: 800)


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

: 

## 5. Advanced: Filtering and Metadata

In [ ]:
# Advanced VectorDB with filtering
class FilterableVectorDB:
    def __init__(self, embedding_model='all-MiniLM-L6xw-v2'):
        self.model = SentenceTransformer(embedding_model)
        self.documents = []
        self.embeddings = None
        self.metadata = []
    
    def add(self, documents: List[str], metadata: List[Dict]):
        self.documents.extend(documents)
        self.metadata.extend(metadata)
        
        new_embeddings = self.model.encode(documents).astype('float32')
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
        print(f"✅ Added {len(documents)} documents")
    
    def search(self, query: str, top_k: int = 5, filters: Dict = None):
        """Search with optional metadata filtering"""
        query_emb = self.model.encode([query]).astype('float32')
        
        # Apply filters
        if filters:
            valid_indices = []
            for i, meta in enumerate(self.metadata):
                match = all(
                    meta.get(key) == value 
                    for key, value in filters.items()
                )
                if match:
                    valid_indices.append(i)
            
            if not valid_indices:
                return []
            
            filtered_embeddings = self.embeddings[valid_indices]
        else:
            valid_indices = list(range(len(self.documents)))
            filtered_embeddings = self.embeddings
        
        # Calculate similarities
        faiss.normalize_L2(query_emb)
        faiss.normalize_L2(filtered_embeddings)
        
        similarities = np.dot(filtered_embeddings, query_emb.T).flatten()
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        # Build results
        results = []
        for rank, idx in enumerate(top_indices):
            orig_idx = valid_indices[idx]
            results.append({
                'rank': rank + 1,
                'score': float(similarities[idx]),
                'document': self.documents[orig_idx],
                'metadata': self.metadata[orig_idx]
            })
        
        return results

# Test filtering
filter_vdb = FilterableVectorDB()

docs_with_meta = [
    ("Python programming basics", {"category": "programming", "level": "beginner"}),
    ("Advanced Python techniques", {"category": "programming", "level": "advanced"}),
    ("Machine learning fundamentals", {"category": "AI", "level": "beginner"}),
    ("Deep learning with PyTorch", {"category": "AI", "level": "advanced"}),
    ("Data structures in Python", {"category": "programming", "level": "intermediate"}),
]

docs = [d[0] for d in docs_with_meta]
meta = [d[1] for d in docs_with_meta]

filter_vdb.add(docs, meta)

# Search with filter
print("\nSearch with Filter (category='AI', level='beginner'):")
results = filter_vdb.search(
    "learning AI",
    top_k=3,
    filters={"category": "AI", "level": "beginner"}
)

for r in results:
    print(f"\n{r['rank']}. [Score: {r['score']:.4f}]")
    print(f"   {r['document']}")
    print(f"   {r['metadata']}")

print("\n💡 Metadata filtering enables precise retrieval!")

## 6. Production RAG with Vector DB

In [ ]:
class ProductionRAGRetriever:
    def __init__(self, 
                 embedding_model='all-MiniLM-L6-v2',
                 index_type='hnsw'):
        self.vdb = VectorDatabase(
            embedding_model=embedding_model,
            index_type=index_type,
            metric='cosine'
        )
        self.chunk_size = 512
        self.chunk_overlap = 50
    
    def chunk_document(self, text: str, metadata: Dict = None):
        """Chunk document with overlap"""
        from nltk.tokenize import sent_tokenize
        import nltk
        nltk.download('punkt', quiet=True)
        
        sentences = sent_tokenize(text)
        chunks = []
        chunk_metadata = []
        
        current_chunk = []
        current_length = 0
        
        for sent in sentences:
            sent_len = len(sent.split())
            
            if current_length + sent_len > self.chunk_size and current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunks.append(chunk_text)
                chunk_metadata.append({
                    **metadata,
                    'chunk_id': len(chunks),
                    'length': current_length
                })
                
                # Keep overlap
                overlap_sents = current_chunk[-2:] if len(current_chunk) > 2 else current_chunk
                current_chunk = overlap_sents + [sent]
                current_length = sum(len(s.split()) for s in current_chunk)
            else:
                current_chunk.append(sent)
                current_length += sent_len
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
            chunk_metadata.append({
                **metadata,
                'chunk_id': len(chunks),
                'length': current_length
            })
        
        return chunks, chunk_metadata
    
    def index_documents(self, documents: List[str], metadata: List[Dict] = None):
        """Index documents with automatic chunking"""
        all_chunks = []
        all_metadata = []
        
        for i, doc in enumerate(documents):
            doc_meta = metadata[i] if metadata else {}
            doc_meta['doc_id'] = i
            
            chunks, chunk_meta = self.chunk_document(doc, doc_meta)
            all_chunks.extend(chunks)
            all_metadata.extend(chunk_meta)
        
        self.vdb.add(all_chunks, all_metadata)
        print(f"📄 Indexed {len(documents)} documents into {len(all_chunks)} chunks")
    
    def retrieve(self, query: str, top_k: int = 5):
        """Retrieve relevant chunks"""
        return self.vdb.search(query, top_k=top_k)
    
    def save(self, path: str):
        self.vdb.save(path)
    
    def load(self, path: str):
        self.vdb.load(path)

# Test production RAG
rag_retriever = ProductionRAGRetriever()

# Sample documents
long_docs = [
    """Retrieval-Augmented Generation (RAG) is a powerful technique that enhances 
    large language models by incorporating external knowledge retrieval. The process 
    works in two main stages: first, relevant documents are retrieved from a knowledge 
    base using similarity search, and then these documents are used as context for 
    the language model to generate informed responses. This approach significantly 
    improves factual accuracy and reduces hallucination.""",
    
    """Vector databases are specialized systems designed to store and query high-dimensional 
    embeddings efficiently. They use advanced indexing techniques like HNSW and IVF to 
    enable fast approximate nearest neighbor search. Popular options include FAISS, 
    ChromaDB, Pinecone, and Weaviate, each with different trade-offs between speed, 
    accuracy, and ease of use."""
]

doc_metadata = [
    {"title": "RAG Introduction", "source": "guide"},
    {"title": "Vector Databases", "source": "guide"}
]

# Index
rag_retriever.index_documents(long_docs, doc_metadata)

# Retrieve
query = "How do vector databases help with RAG?"
results = rag_retriever.retrieve(query, top_k=3)

print(f"\nQuery: '{query}'\n")
print("Retrieved Chunks:")
for r in results:
    print(f"\n{r['rank']}. [Score: {r['score']:.4f}]")
    print(f"   {r['document'][:100]}...")
    print(f"   Metadata: {r['metadata']}")

## Key Takeaways 🎯

### ✅ Vector Database Essentials:

1. **Purpose**: Fast similarity search at scale
2. **ANN Algorithms**: HNSW, IVF, LSH for speed
3. **Trade-offs**: Speed ↔ Accuracy ↔ Memory
4. **Metadata**: Essential for filtering/routing
5. **Persistence**: Save/load indexes for production

### 🗄️ Database Selection Guide:

| Database | Best For | Pros | Cons |
|----------|----------|------|------|
| **FAISS** | Max speed | Fastest, flexible | No metadata filtering |
| **ChromaDB** | Prototyping | Easy, embedded | Limited scale |
| **Pinecone** | Production | Managed, scalable | Cost |
| **Weaviate** | Features | Rich features | Complex setup |
| **Qdrant** | Performance | Fast, modern | Newer |

### 🔧 Index Types:

```python
# Flat (Exact) - Best accuracy, slower
IndexFlatL2 / IndexFlatIP
- Use for: <100k vectors
- Accuracy: 100%
- Speed: O(n)

# HNSW - Best balance
IndexHNSWFlat
- Use for: <10M vectors  
- Accuracy: 95-99%
- Speed: O(log n)

# IVF - Best for huge datasets
IndexIVFFlat
- Use for: >10M vectors
- Accuracy: 90-95%
- Speed: O(√n)
```

### 📊 Similarity Metrics:

| Metric | Use When | Formula | Range |
|--------|----------|---------|-------|
| **Cosine** | Normalized vectors | dot(a,b)/(‖a‖‖b‖) | [-1, 1] |
| **Dot Product** | Pre-normalized | dot(a,b) | (-∞, ∞) |
| **L2 (Euclidean)** | Absolute distance | ‖a-b‖ | [0, ∞) |

### 💡 Production Best Practices:

1. **Choose the right index**:
   ```python
   # <100k docs: Flat (exact)
   # 100k-10M: HNSW
   # >10M: IVF or cloud solution
   ```

2. **Always normalize for cosine**:
   ```python
   faiss.normalize_L2(embeddings)
   # Then use IndexFlatIP (inner product)
   ```

3. **Chunk documents properly**:
   ```python
   - Size: 256-512 tokens
   - Overlap: 50-100 tokens
   - Preserve sentence boundaries
   ```

4. **Use metadata for filtering**:
   ```python
   metadata = {
       'source': 'docs',
       'category': 'technical',
       'date': '2024-01',
       'chunk_id': 0
   }
   ```

5. **Save indexes to disk**:
   ```python
   faiss.write_index(index, 'index.faiss')
   # Avoid re-indexing in production!
   ```

### 🚀 Scaling Strategies:

```python
# Small (<1M docs)
- Local: FAISS, ChromaDB
- Index: HNSW
- Storage: Disk

# Medium (1M-100M docs)  
- Managed: Pinecone, Qdrant
- Index: IVF
- Storage: Distributed

# Large (>100M docs)
- Cloud: Pinecone, Weaviate Cloud
- Index: IVF + PQ (compression)
- Storage: Sharded
```

### ⚡ Performance Tuning:

```python
# HNSW Parameters
M = 32  # Connections (16-64)
ef_construction = 40  # Build quality (40-200)
ef_search = 16  # Search quality (10-100)

# Trade-offs:
- Higher M → Better quality, more memory
- Higher ef → Better quality, slower
- Lower values → Faster, less accurate
```

### 🎯 Quick Decision Tree:

```
How many documents?
├─ <100k → FAISS Flat (exact)
├─ 100k-1M → FAISS HNSW
├─ 1M-10M → ChromaDB or Weaviate
└─ >10M → Pinecone or Qdrant Cloud

Need metadata filtering?
├─ Yes → ChromaDB, Weaviate, Pinecone
└─ No → FAISS (fastest)

Budget?
├─ Free → FAISS, ChromaDB (local)
└─ Paid → Pinecone, Weaviate Cloud
```

## Next Steps 🔜

Move to `05_Reranking_Techniques.ipynb` for advanced optimization!

Learn to boost retrieval quality with cross-encoders and multi-stage retrieval! 🎯